In [27]:
import json
import re

# Conventional commit pattern and semantic mapping
CONVENTIONAL_PATTERN = re.compile(
    r'^(?P<type>\w+)(?:\((?P<scope>[^)]+)\))?(?:!)?:\s*(?P<description>.+)$'
)

TASK_MAPPING = {
    "feat": "feature_development", "fix": "defect_resolution",
    "refactor": "code_restructuring", "docs": "documentation",
    "test": "quality_assurance", "build": "build_engineering",
    "ci": "deployment_engineering", "chore": "maintenance",
    "perf": "performance_improvement", "style": "formatting"
}


def get_attr(attrs, name):
    for attr in attrs:
        if attr["name"] == name:
            return attr.get("value")
    return None


def set_attr(attrs, name, value):
    if value is None:
        return
    for attr in attrs:
        if attr["name"] == name:
            attr["value"] = value
            return
    attrs.append({"name": name, "value": value})


def parse_commit(message):
    if not message:
        return None
    m = CONVENTIONAL_PATTERN.match(message.strip())
    if not m:
        return None
    t = m.group("type").lower()
    return {
        "task_type": t,
        "task_scope": m.group("scope"),
        "task_description": m.group("description"),
        "task_semantic_class": TASK_MAPPING.get(t, "other")
    }


In [28]:
# Load and enrich OCEL with task objects
with open("commitizen.json") as f:
    ocel = json.load(f)

objects = ocel.get("objects", [])
commit_to_task = {}
task_counter = 0
enriched_commits = 0

# Step 1: Enrich commits with task metadata from messages
for obj in objects:
    if obj.get("type") != "commit":
        continue
    
    attrs = obj.setdefault("attributes", [])
    message = get_attr(attrs, "commit_message")
    parsed = parse_commit(message)
    
    if parsed:
        for k, v in parsed.items():
            set_attr(attrs, k, v)
        enriched_commits += 1

# Step 2: Create task objects and link to commits + events
obj_types = ocel.setdefault("objectTypes", [])
if not any(t["name"] == "task" for t in obj_types):
    obj_types.append({
        "name": "task",
        "attributes": [
            {"name": "task_type", "type": "string"},
            {"name": "task_scope", "type": "string"},
            {"name": "task_description", "type": "string"},
            {"name": "task_semantic_class", "type": "string"}
        ]
    })

new_tasks = []
for obj in objects:
    if obj.get("type") != "commit":
        continue
    
    attrs = obj.setdefault("attributes", [])
    message = get_attr(attrs, "commit_message")
    parsed = parse_commit(message)
    
    if not parsed:
        continue
    
    task_id = f"task_{task_counter}"
    task_counter += 1
    
    # Create task object
    task_obj = {
        "id": task_id,
        "type": "task",
        "attributes": [
            {"name": "task_type", "value": parsed["task_type"]},
            {"name": "task_scope", "value": parsed["task_scope"]},
            {"name": "task_description", "value": parsed["task_description"]},
            {"name": "task_semantic_class", "value": parsed["task_semantic_class"]}
        ],
        "relationships": []
    }
    new_tasks.append(task_obj)
    
    # Link commit to task
    obj.setdefault("relationships", []).append({"objectId": task_id, "qualifier": "realizes"})
    commit_to_task[obj["id"]] = task_id

# Link tasks to events
events = ocel.get("events", [])
events_with_tasks = 0
for event in events:
    event_rels = event.get("relationships", [])
    commit_ids = {rel["objectId"] for rel in event_rels if rel["objectId"] in commit_to_task}
    
    for commit_id in commit_ids:
        task_id = commit_to_task[commit_id]
        if not any(rel["objectId"] == task_id for rel in event_rels):
            event_rels.append({"objectId": task_id, "qualifier": "realizes"})
            events_with_tasks += 1

objects.extend(new_tasks)

# Save enriched OCEL
with open("commitizen_task_objects.json", "w") as f:
    json.dump(ocel, f, indent=2)

print(f"Enriched {enriched_commits} commits")
print(f"Created {len(new_tasks)} task objects")
print(f"Linked {events_with_tasks} event-task relationships")


Enriched 1721 commits
Created 1721 task objects
Linked 1721 event-task relationships
